In [119]:
import polars as pl
import pandas as pd
import numpy as np
import os

DATA_PATH = r".././dataset/sales_pers.item_chunk_0.parquet"

df = pl.read_parquet(DATA_PATH)
print(f"Số dòng: {df.height:,}, Số cột: {df.width}")


Số dòng: 27,332, Số cột: 34


In [120]:
print("\nCác cột ban đầu:")
print(df.columns)


Các cột ban đầu:
['p_id', 'item_id', 'price', 'category_l1_id', 'category_l1', 'category_l2_id', 'category_l2', 'category_l3_id', 'category_l3', 'category_id', 'category', 'description', 'brand', 'manufacturer', 'creation_timestamp', 'is_deleted', 'created_date', 'updated_date', 'sync_status_id', 'last_sync_date', 'sync_error_message', 'image_url', 'gender_target', 'age_group', 'item_type', 'gp', 'weight', 'color', 'size', 'origin', 'volume', 'material', 'sale_status', 'description_new']


# Task 1: Loại bỏ các cột mà nhóm nghĩ là không cần thiết.

In [121]:
# --- Danh sách cột cần loại bỏ (theo phân tích EDA & tương quan) ---
cols_to_drop = [
    # --- Metadata hệ thống (không dùng cho mô hình) ---
    "p_id",
    "is_deleted",
    "sync_status_id",
    "sync_error_message",
    "image_url",
    "last_sync_date",
    "creation_timestamp",
    "updated_date",
    "created_date",
    "sale_status",

    # --- Cột numeric hầu như vô nghĩa / chỉ có 1 giá trị ---
    "gp",
    "weight",
    "volume",

    # --- Các cột chất lượng kém / missing cực cao / không dùng ---
    "color",
    "size",
    "material",
    "origin",
    "manufacturer",

    # --- ID phân loại (trùng với tên category) ---
    "category_l1_id",
    "category_l2_id",
    "category_l3_id",
    "category_id"
]



# --- Loại bỏ các cột không cần thiết ---
df_cleaned = df.drop(cols_to_drop)

print(f"\nĐã loại bỏ {len(cols_to_drop)} cột không cần thiết.")
print(f"Số cột còn lại: {df_cleaned.width}")
print("\nDanh sách cột sau khi loại bỏ:")
print(df_cleaned.columns)



Đã loại bỏ 22 cột không cần thiết.
Số cột còn lại: 12

Danh sách cột sau khi loại bỏ:
['item_id', 'price', 'category_l1', 'category_l2', 'category_l3', 'category', 'description', 'brand', 'gender_target', 'age_group', 'item_type', 'description_new']


# Task 2: Xử lý NULL, Xử lý Outlier

## Xử lý outlier

In [122]:
# df_cleaned: dataframe sau khi đã drop các cột không cần thiết
print("Số lượng ban đầu:", df_cleaned.height)

# --- LOẠI SẢN PHẨM CÓ PRICE ≤ 1 (Outlier) ---
df_no_outlier = df_cleaned.filter(pl.col("price") > 1)

print("Số lượng sau khi loại outlier (price ≤ 1):", df_no_outlier.height)
print("Số lượng bị loại:", df_cleaned.height - df_no_outlier.height)

# Kiểm tra lại xem còn outlier hay không
print("\nKiểm tra min price sau khi xử lý:")
print(df_no_outlier.select(pl.col("price").min()))

Số lượng ban đầu: 27332
Số lượng sau khi loại outlier (price ≤ 1): 27323
Số lượng bị loại: 9

Kiểm tra min price sau khi xử lý:
shape: (1, 1)
┌───────────────┐
│ price         │
│ ---           │
│ decimal[38,4] │
╞═══════════════╡
│ 1000.0000     │
└───────────────┘


In [123]:
# Chuẩn hóa giá trị Unisex → Không xác định
df_no_outlier = df_no_outlier.with_columns(
    pl.col("gender_target").replace("Unisex", "Không xác định")
)

print("Đã chuyển toàn bộ Unisex thành 'Không xác định'.")
print(df_no_outlier["gender_target"].value_counts())


Đã chuyển toàn bộ Unisex thành 'Không xác định'.
shape: (4, 2)
┌────────────────┬───────┐
│ gender_target  ┆ count │
│ ---            ┆ ---   │
│ str            ┆ u32   │
╞════════════════╪═══════╡
│ Bé Trai        ┆ 3318  │
│ Bé Gái         ┆ 4108  │
│ Sơ sinh        ┆ 1862  │
│ Không xác định ┆ 18035 │
└────────────────┴───────┘


In [124]:
df_no_outlier.head(5)

item_id,price,category_l1,category_l2,category_l3,category,description,brand,gender_target,age_group,item_type,description_new
str,"decimal[38,4]",str,str,str,str,str,str,str,str,str,str
"""0502020000004""",99000.0000,"""Babycare""","""Bình sữa, phụ kiện""","""Núm ty""","""Núm ty Dr Brown""","""Không xác định""","""Dr.Brown's""","""Không xác định""","""Không xác định""","""Không xác định""","""Chi tiết sản phẩm Tên sản phẩm…"
"""0010290040150""",69000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Bộ quần áo bé gái""","""Không xác định""","""Con Cưng""","""Bé Gái""","""Từ 3Y""","""Bộ quần áo""","""Không xác định"""
"""0008010000015""",45000.0000,"""Đồ chơi & Sách""","""0-1Y""","""Gặm nướu""","""Gặm nướu khác""","""- Chất liệu: Sản phẩm được làm bằng chất liệu sili…","""Thương hiệu khác""","""Không xác định""","""Không xác định""","""Không xác định""","""Chi tiết sản phẩm Tên sản phẩm…"
"""0020010000094""",401000.0000,"""Tã""","""Merries""","""Merries""","""Merries_Sơ Sinh""","""﻿﻿Tã dán Merries size S 82 miếng là sản phẩm dành …","""Merries Nhật""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định"""
"""0020010000098""",401000.0000,"""Tã""","""Merries""","""Merries""","""Merries_Tã Quần""","""﻿﻿﻿Bỉm tã quần Merries size M 58 miếng là sản phẩm…","""Merries Nhật""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định"""


## Xử lý null

In [125]:
pl.Config.set_tbl_rows(50)             # số dòng tối đa hiển thị


# df_final = df_no_outlier
df_final = df_no_outlier

n = df_final.height

results = []

for col in df_final.columns:
    s = df_final[col]

    # NULL
    null_count = s.null_count()

    # UNKNOWN ("Không xác định")
    if s.dtype == pl.String:
        # Đếm trực tiếp bằng Python list, không lỗi
        unknown_count = s.to_list().count("Không xác định")
    else:
        unknown_count = 0

    results.append({
        "column": col,
        "dtype": s.dtype,
        "null_count": null_count,
        "null_ratio(%)": round(null_count / n * 100, 2),
        "unknown_count": unknown_count,
        "unknown_ratio(%)": round(unknown_count / n * 100, 2)
    })

df_missing_report = pl.DataFrame(results)

print(df_missing_report)


shape: (12, 6)
┌─────────────────┬─────────────────┬────────────┬───────────────┬───────────────┬─────────────────┐
│ column          ┆ dtype           ┆ null_count ┆ null_ratio(%) ┆ unknown_count ┆ unknown_ratio(% │
│ ---             ┆ ---             ┆ ---        ┆ ---           ┆ ---           ┆ )               │
│ str             ┆ object          ┆ i64        ┆ f64           ┆ i64           ┆ ---             │
│                 ┆                 ┆            ┆               ┆               ┆ f64             │
╞═════════════════╪═════════════════╪════════════╪═══════════════╪═══════════════╪═════════════════╡
│ item_id         ┆ String          ┆ 0          ┆ 0.0           ┆ 0             ┆ 0.0             │
│ price           ┆ Decimal(precisi ┆ 0          ┆ 0.0           ┆ 0             ┆ 0.0             │
│                 ┆ on=38, scale=4) ┆            ┆               ┆               ┆                 │
│ category_l1     ┆ String          ┆ 0          ┆ 0.0           ┆ 0        

### Xử lý gender_target

In [126]:

# Lọc các item thời trang nhưng gender_target = "Không xác định"
df_fashion_unknown = (
    df_no_outlier
        .filter(
            (pl.col("category_l1") == "Thời trang") &
            (pl.col("gender_target") == "Không xác định")
        )
)

print("Tổng số item thời trang có gender_target = 'Không xác định':", df_fashion_unknown.height)

# Lấy 20 dòng ngẫu nhiên
df_sample = df_fashion_unknown.sample(n=20, with_replacement=False)

pd.set_option('display.max_columns', None)
df_sample.head(10)

Tổng số item thời trang có gender_target = 'Không xác định': 7264


item_id,price,category_l1,category_l2,category_l3,category,description,brand,gender_target,age_group,item_type,description_new
str,"decimal[38,4]",str,str,str,str,str,str,str,str,str,str
"""3532000000003""",140000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Quần, áo & phụ kiện sơ sinh cũ""","""Quần sơ sinh""","""Không xác định""","""Nous""","""Không xác định""","""Từ 6M""","""Quần""","""Không xác định"""
"""3390000000107""",199000.0000,"""Thời trang""","""Modal kháng khuẩn""","""Bodysuit Modal""","""Bodysuit Modal lẻ""","""Không xác định""","""Animo""","""Không xác định""","""9M-12M""","""Bodysuit""","""Không xác định"""
"""0863004760007""",49000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Áo bé gái""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""3532000000273""",119000.0000,"""Thời trang""","""Quần áo & Phụ kiện sơ sinh""","""Quần""","""Quần sơ sinh Animo""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""6048000000012""",249000.0000,"""Thời trang""","""Thời trang đông""","""Bodysuit đông""","""0-12M Bodysuit đông vải mỏng""","""Không xác định""","""Animo""","""Không xác định""","""6M-9M""","""Bodysuit""","""Không xác định"""
"""3532003170001""",49000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Quần, áo & phụ kiện sơ sinh cũ""","""Quần sơ sinh""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""3953000000721""",129000.0000,"""Thời trang""","""Thời trang bé trai""","""Bộ bé trai""","""Bộ bé trai Animo Easy""","""Không xác định""","""Animo""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định"""
"""3481104360001""",19000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Quần, áo & phụ kiện sơ sinh cũ""","""Nón sơ sinh""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""3522095180001""",49000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Quần, áo & phụ kiện sơ sinh cũ""","""Áo sơ sinh""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null


In [127]:
# Lọc sản phẩm thời trang có category chứa "đầm" hoặc "váy" và gender_target = "Không xác định"
df_dam_vay_unknown = (
    df_no_outlier
    .filter(
        (pl.col("category_l1") == "Thời trang")
        &
        (pl.col("gender_target") == "Không xác định")
        &
        (
            pl.col("category").str.contains("đầm", literal=False)
            | pl.col("category").str.contains("váy", literal=False)
        )
    )
)

# In thống kê số dòng
print("Tổng sản phẩm thỏa điều kiện:", df_dam_vay_unknown.height)

# Lấy mẫu 20 dòng
df_sample = df_dam_vay_unknown.sample(n=20, seed=42)

# Hiển thị đầy đủ cột
pd.set_option('display.max_columns', None)
df_sample.head(10)


Tổng sản phẩm thỏa điều kiện: 192


item_id,price,category_l1,category_l2,category_l3,category,description,brand,gender_target,age_group,item_type,description_new
str,"decimal[38,4]",str,str,str,str,str,str,str,str,str,str
"""6045000000002""",349000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Quần, chân váy bé gái""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""1092093850001""",189000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Đồ bầu""","""Quần, áo, đầm bầu""","""Không xác định""","""CF (ConCung Fashion)""","""Không xác định""","""Mẹ""","""Áo bầu""","""Không xác định"""
"""1083022850002""",149000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Đồ bầu""","""Quần, áo, đầm bầu""","""Không xác định""","""CF (ConCung Fashion)""","""Không xác định""","""Mẹ""","""Quần bầu""","""Không xác định"""
"""0886026770002""",49000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Quần, chân váy bé gái""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""0887026770004""",49000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Quần, chân váy bé gái""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""0891019790002""",49000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Quần, chân váy bé gái""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""0012190160065""",49000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Quần, chân váy bé gái""","""Không xác định""","""Laluna""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định"""
"""1080033860001""",199000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Đồ bầu""","""Quần, áo, đầm bầu""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""1080004860001""",189000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Đồ bầu""","""Quần, áo, đầm bầu""","""Không xác định""","""CF (ConCung Fashion)""","""Không xác định""","""Mẹ""","""Áo bầu""","""Không xác định"""


In [128]:
# ============================================
#  TẠO CỘT gender_target_final
# ============================================

df_filled = df_no_outlier.with_columns(
    pl.col("gender_target").alias("gender_target_final")
)

# ============================================
#  1) FILL “SƠ SINH”
# ============================================

mask_sosinh = (
    (pl.col("category_l1") == "Thời trang")
    & (pl.col("gender_target_final") == "Không xác định")
    & (
        pl.col("category_l2").str.contains("sơ sinh", literal=False)
        | pl.col("category").str.contains("sơ sinh", literal=False)
        | pl.col("category").str.contains("0-3m", literal=False)
        | pl.col("category").str.contains("3-6m", literal=False)
        | pl.col("category").str.contains("0-12m", literal=False)
        | pl.col("category").str.contains(r"\bnb\b", literal=False)   # NB
        | pl.col("category").str.contains("newborn", literal=False)
    )
)

df_filled = df_filled.with_columns(
    pl.when(mask_sosinh)
      .then(pl.lit("Sơ sinh"))
      .otherwise(pl.col("gender_target_final"))
      .alias("gender_target_final")
)

# ============================================
#  2) FILL “BÉ GÁI”
# ============================================

mask_fashion_unknown = (
    (pl.col("category_l1") == "Thời trang")
    & (pl.col("gender_target_final") == "Không xác định")
)

mask_not_maternity = (
    (~pl.col("category").str.contains("bầu", literal=False))
    & (~pl.col("category_l2").str.contains("bầu", literal=False))
    & (~pl.col("category_l3").str.contains("bầu", literal=False))
)

mask_not_ambiguous = (
    ~pl.col("category_l3").str.contains("Thời trang bé trai, bé gái cũ", literal=False)
)

mask_girl_keyword = (
    pl.col("category").str.contains("bé gái|đầm|váy|chân váy", literal=False)
    | pl.col("category_l2").str.contains("bé gái|đầm bé gái|bộ bé gái", literal=False)
    | pl.col("category_l3").str.contains("bé gái|đầm|váy", literal=False)
)

mask_fill_girl = (
    mask_fashion_unknown
    & mask_not_maternity
    & mask_not_ambiguous
    & mask_girl_keyword
)

df_filled = df_filled.with_columns(
    pl.when(mask_fill_girl)
      .then(pl.lit("Bé Gái"))
      .otherwise(pl.col("gender_target_final"))
      .alias("gender_target_final")
)

# ============================================
#  3) FILL “BÉ TRAI”
# ============================================

mask_boy_keyword = (
    pl.col("category").str.contains("bé trai", literal=False)
    | pl.col("category_l2").str.contains("bé trai|bộ bé trai", literal=False)
    | pl.col("category_l3").str.contains("bé trai", literal=False)
)

mask_fill_boy = (
    mask_fashion_unknown
    & mask_not_ambiguous
    & mask_boy_keyword
)

df_filled = df_filled.with_columns(
    pl.when(mask_fill_boy)
      .then(pl.lit("Bé Trai"))
      .otherwise(pl.col("gender_target_final"))
.alias("gender_target_final")
)

# ============================================
# 4) KIỂM TRA KẾT QUẢ FILL
# ============================================

print("\nTổng số dòng được fill:",
      df_filled.filter(pl.col("gender_target") != pl.col("gender_target_final")).height
)


total_rows = df_filled.height

unknown_original = df_filled.filter(
    pl.col("gender_target") == "Không xác định"
).height

unknown_final = df_filled.filter(
    pl.col("gender_target_final") == "Không xác định"
).height

ratio_unknown_final = unknown_final / total_rows * 100

print("\n=== THỐNG KÊ 'Không xác định' ===")
print(f"1) gender_target ban đầu = 'Không xác định': {unknown_original:,}")
print(f"2) gender_target_final    = 'Không xác định': {unknown_final:,}")
print(f"3) Tỷ lệ 'Không xác định' sau fill: {ratio_unknown_final:.2f}% (trên {total_rows:,} dòng)")

# Xuất ra 5 dòng mẫu sau khi fill
print("\n=== 5 dòng mẫu đã được fill ===")
print(
    df_filled
    .filter(
        (pl.col("gender_target") == "Không xác định")
        & (pl.col("gender_target_final") != "Không xác định")
    )
    .head(5)
)


Tổng số dòng được fill: 3333

=== THỐNG KÊ 'Không xác định' ===
1) gender_target ban đầu = 'Không xác định': 18,035
2) gender_target_final    = 'Không xác định': 14,702
3) Tỷ lệ 'Không xác định' sau fill: 53.81% (trên 27,323 dòng)

=== 5 dòng mẫu đã được fill ===
shape: (5, 13)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ item_id   ┆ price     ┆ category_ ┆ category_ ┆ … ┆ age_group ┆ item_type ┆ descripti ┆ gender_t │
│ ---       ┆ ---       ┆ l1        ┆ l2        ┆   ┆ ---       ┆ ---       ┆ on_new    ┆ arget_fi │
│ str       ┆ decimal[3 ┆ ---       ┆ ---       ┆   ┆ str       ┆ str       ┆ ---       ┆ nal      │
│           ┆ 8,4]      ┆ str       ┆ str       ┆   ┆           ┆           ┆ str       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 603400000 ┆

In [129]:

# Lọc các sản phẩm thuộc Phụ kiện và gender_target = "Không xác định"
df_phukien_unknown = (
    df_no_outlier
    .filter(
        (pl.col("category_l1") == "Phụ kiện") &
        (pl.col("gender_target") == "Không xác định")
    )
)

# Kiểm tra số lượng
print("Tổng số sản phẩm Phụ kiện có gender_target = 'Không xác định':",
      df_phukien_unknown.height)

# Lấy ngẫu nhiên 10 dòng
df_sample = df_phukien_unknown.sample(n=10, seed=42)

# In ra toàn bộ cột để bạn xem xét thật chi tiết
pd.set_option('display.max_columns', None)
df_sample.head(10)


Tổng số sản phẩm Phụ kiện có gender_target = 'Không xác định': 1666


item_id,price,category_l1,category_l2,category_l3,category,description,brand,gender_target,age_group,item_type,description_new
str,"decimal[38,4]",str,str,str,str,str,str,str,str,str,str
"""6053000000007""",49000.0000,"""Phụ kiện""","""Cơ cấu hàng cũ""","""Giày dép tồn""","""Giày bún tập đi""","""﻿Giày bún tập đi Animo A2206_JK010 với chất liệu đ…","""Animo""","""Không xác định""","""1Y-3Y""","""Giày""","""Chi tiết sản phẩm Tên sản phẩm…"
"""6055000000002""",49000.0000,"""Phụ kiện""","""Cơ cấu hàng cũ""","""Giày dép tồn""","""Giày bún tập đi""","""Không xác định""","""Animo""","""Không xác định""","""Không xác định""","""Giày""","""Chi tiết sản phẩm Tên sản phẩm…"
"""4804000000016""",249000.0000,"""Phụ kiện""","""Phụ kiện khác""","""Túi xách, Ba lô""","""Ba lô""","""Không xác định""","""Mesuca""","""Không xác định""","""Từ 2Y""","""Balo""","""Chi tiết sản phẩm Tên sản phẩm…"
"""4794000000015""",139000.0000,"""Phụ kiện""","""Cơ cấu hàng cũ""","""Phụ kiện tồn""","""Phụ kiện tồn SPC""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""4812000000011""",99000.0000,"""Phụ kiện""","""Cơ cấu hàng cũ""","""Giày dép tồn""","""Dép sục người lớn""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""6055000000036""",119000.0000,"""Phụ kiện""","""Giày tập đi""","""Giày sơ sinh 119k""","""Giày sơ sinh 119k S11""","""Không xác định""","""Animo""","""Không xác định""","""Không xác định""","""Không xác định""","""Chi tiết sản phẩm Tên sản phẩm…"
"""0014521040104""",9000.0000,"""Phụ kiện""","""Cơ cấu hàng cũ""","""Phụ kiện tồn""","""Phụ kiện khác tồn""","""Không xác định""","""CF (ConCung Fashion)""","""Không xác định""","""Không xác định""","""Kẹp tóc""","""Chi tiết sản phẩm Tên sản phẩm…"
"""4003000000014""",229000.0000,"""Phụ kiện""","""Giày tập đi""","""Giày chút chít 179k""","""Giày chút chít 229k S15""","""﻿﻿Giày tập đi chút chít Animo A2204_MN001 (14-17,H…","""Animo""","""Không xác định""","""0-24M""","""Giày tập đi""","""Chi tiết sản phẩmTên sản phẩm: Giày tập đi chút ch…"
"""4027000000049""",139000.0000,"""Phụ kiện""","""Nón""","""Nón sơ sinh""","""0-12M Nón khăn voan""","""﻿﻿Nón vành tròn khăn voan bé trai Animo A2410_MN02…","""Animo""","""Không xác định""","""0-12M""","""Nón""","""Chi tiết sản phẩm Tên sản phẩm…"


### Xử lý age_group

In [130]:
# Lọc các sản phẩm Thời trang nhưng age_group = "Không xác định"
df_fashion_age_unknown = (
    df_no_outlier
    .filter(
        (pl.col("category_l1") == "Thời trang") &
        (pl.col("age_group") == "Không xác định")
    )
)

# Kiểm tra số lượng
total = df_fashion_age_unknown.height
print("Tổng số sản phẩm Thời trang có age_group = 'Không xác định':", total)

# -----------------------------
# Thống kê description & description_new
# -----------------------------

count_desc_unknown = df_fashion_age_unknown.filter(pl.col("description") == "Không xác định").height
count_desc_new_unknown = df_fashion_age_unknown.filter(pl.col("description_new") == "Không xác định").height

print("\nThống kê tình trạng mô tả trong nhóm này:")
print(f"- description = 'Không xác định': {count_desc_unknown} / {total} ({count_desc_unknown/total*100:.2f}%)")
print(f"- description_new = 'Không xác định': {count_desc_new_unknown} / {total} ({count_desc_new_unknown/total*100:.2f}%)")

# -----------------------------
# Lấy ngẫu nhiên 20 dòng để quan sát
# -----------------------------
df_sample_age = df_fashion_age_unknown.sample(n=20, seed=42)

pd.set_option('display.max_columns', None)
df_sample_age.head(20)


Tổng số sản phẩm Thời trang có age_group = 'Không xác định': 6058

Thống kê tình trạng mô tả trong nhóm này:
- description = 'Không xác định': 5435 / 6058 (89.72%)
- description_new = 'Không xác định': 2593 / 6058 (42.80%)


item_id,price,category_l1,category_l2,category_l3,category,description,brand,gender_target,age_group,item_type,description_new
str,"decimal[38,4]",str,str,str,str,str,str,str,str,str,str
"""3523000000021""",99000.0000,"""Thời trang""","""Quần áo & Phụ kiện sơ sinh""","""Áo""","""Áo sơ sinh Animo""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""3320016830007""",69000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Bộ quần áo bé trai""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""0930002360001""",19000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Quần, áo & phụ kiện sơ sinh cũ""","""Tã vải""","""Không xác định""","""CF (ConCung Fashion)""","""Sơ sinh""","""Không xác định""","""Tã vải""","""Không xác định"""
"""3533000000241""",119000.0000,"""Thời trang""","""Quần áo & Phụ kiện sơ sinh""","""Quần""","""Quần sơ sinh Animo""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""6034000000360""",129000.0000,"""Thời trang""","""Thời trang bé gái""","""Bộ bé gái""","""Bộ bé gái Animo Easy""","""Không xác định""","""Animo""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định"""
"""3951000000444""",139000.0000,"""Thời trang""","""Thời trang bé trai""","""Bộ bé trai""","""Bộ bé trai Animo Easy""","""﻿﻿Bộ bé trai ngắn Animo Easy HN0725030 sở hữu thiế…","""Animo""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định"""
"""0891104050001""",49000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Quần, chân váy bé gái""","""Không xác định""","""CF (ConCung Fashion)""","""Bé Gái""","""Không xác định""","""Quần""","""Không xác định"""
"""6998000000086""",379000.0000,"""Thời trang""","""Modal kháng khuẩn""","""Bộ Modal""","""Bộ chống muỗi set 2""","""Không xác định""","""Animo""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định"""
"""3414016830002""",89000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Bộ quần áo bé trai""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null


#### Thống kê `age_group` xuất hiện trong `category_l1` (top 15 `age_group`)

In [131]:
import polars as pl

# Các category_l1 muốn thống kê riêng
target_cat_l1 = [
    "Babycare",
    "Thời trang",
    "Đồ chơi & Sách",
    "Tã",
    "Textile",
    "Phụ kiện",
    "Thực phẩm cho bé",        # nhớ check lại đúng tên trong data
    "Sữa",
    "Hóa mỹ phẩm cho bé",      # check lại spelling "Hóa/Hoá"
    "Thực phẩm cho gia đình",  # check lại spelling
    "Vệ sinh",
    "Sữa nước",
    "TPCN",
]

# Lặp qua từng category_l1 và in bảng thống kê age_group cho riêng category đó
for cat in target_cat_l1:
    print(f"\n================= {cat} =================")

    sub_stats = (
        df_filled
        .filter(pl.col("category_l1") == cat)
        .group_by("age_group")
        .agg(pl.len().alias("count"))
        .sort("count", descending=True)
        .head(15)          # giữ top 15 age_group của riêng category này
    )

    print(sub_stats)


================= Babycare =================
shape: (15, 2)
┌────────────────┬───────┐
│ age_group      ┆ count │
│ ---            ┆ ---   │
│ str            ┆ u32   │
╞════════════════╪═══════╡
│ Không xác định ┆ 1667  │
│ Từ 6M          ┆ 59    │
│ Mẹ             ┆ 37    │
│ Từ 0M          ┆ 37    │
│ Từ 1Y          ┆ 23    │
│ 0-6M           ┆ 19    │
│ Từ 3Y          ┆ 15    │
│ Từ 3M          ┆ 13    │
│ Từ 2Y          ┆ 11    │
│ 3M-6M          ┆ 10    │
│ 0-3M           ┆ 9     │
│ 2M-6M          ┆ 9     │
│ 0-36M          ┆ 9     │
│ Từ 4M          ┆ 8     │
│ Từ 9M          ┆ 7     │
└────────────────┴───────┘

================= Thời trang =================
shape: (15, 2)
┌────────────────┬───────┐
│ age_group      ┆ count │
│ ---            ┆ ---   │
│ str            ┆ u32   │
╞════════════════╪═══════╡
│ Không xác định ┆ 6058  │
│ 9M-12M         ┆ 844   │
│ 6M-9M          ┆ 671   │
│ 3M-6M          ┆ 535   │
│ Từ 2Y          ┆ 493   │
│ Từ 1Y          ┆ 438   │
│ 12M-18M   

#### Kiểm tra `age_group` trong `"Thời trang"`

In [132]:
import polars as pl

CAT_L1 = "Thời trang"

df_fashion = df_filled.filter(pl.col("category_l1") == CAT_L1)

# 1) Tất cả class trong category_l2
cat2_classes = (
    df_fashion
    .select("category_l2")
    .unique()                 # lấy giá trị duy nhất
    .sort("category_l2")      # sắp xếp cho dễ nhìn (optional)
)

print("\n=== Tất cả class category_l2 của Thời trang ===")
print(cat2_classes)

# 2) Tất cả class trong category_l3
cat3_classes = (
    df_fashion
    .select("category_l3")
    .unique()
    .sort("category_l3")
)

print("\n=== Tất cả class category_l3 của Thời trang ===")
print(cat3_classes)

# 3) Tất cả class trong category (tên chi tiết)
cat_classes = (
    df_fashion
    .select("category")
    .unique()
    .sort("category")
)

print("\n=== Tất cả class category (tên hiển thị) của Thời trang ===")
print(cat_classes)


=== Tất cả class category_l2 của Thời trang ===
shape: (7, 1)
┌────────────────────────────┐
│ category_l2                │
│ ---                        │
│ str                        │
╞════════════════════════════╡
│ Cho mẹ                     │
│ Cơ cấu hàng cũ             │
│ Modal kháng khuẩn          │
│ Quần áo & Phụ kiện sơ sinh │
│ Thời trang bé gái          │
│ Thời trang bé trai         │
│ Thời trang đông            │
└────────────────────────────┘

=== Tất cả class category_l3 của Thời trang ===
shape: (24, 1)
┌────────────────────────────────┐
│ category_l3                    │
│ ---                            │
│ str                            │
╞════════════════════════════════╡
│ Bao tay chân, nón              │
│ Bodysuit                       │
│ Bodysuit Modal                 │
│ Bodysuit bé gái                │
│ Bodysuit bé trai               │
│ Bodysuit đông                  │
│ Bộ Modal                       │
│ Bộ bé gái                      │
│ Bộ bé trai   

In [133]:
categories = cat_classes.get_column("category").to_list()

print(f"Tổng số class category trong Thời trang: {len(categories)}\n")

for c in categories:
    print(c)

Tổng số class category trong Thời trang: 90

0-12M Bodysuit bé trai đùi
0-12M Bodysuit tam giác
0-12M Bodysuit đông vải mỏng
0-12M Bộ nón tay chân sơ sinh bo
0-12M Quần sơ sinh ngắn
0-12M Áo sơ sinh cài chéo tay ngắn
0-24M Áo thun bé trai tay dài đông
0-3Y Bộ đông bé trai dài
Bao tay chân
Bodysuit
Bodysuit Animo
Bodysuit Animo Easy
Bodysuit Chống muỗi set 2
Bodysuit Modal lẻ
Bodysuit Modal set 2
Bodysuit bé gái dài
Bodysuit bé gái tam giác
Bodysuit bé gái đùi
Bodysuit bé trai dài
Bodysuit bé trai tam giác
Bodysuit bé trai đùi
Bodysuit cũ
Bodysuit tam giác
Bodysuit đông vải dày
Bodysuit đông vải mỏng
Bộ Modal lẻ
Bộ Modal set 2
Bộ bé gái Animo
Bộ bé gái Animo Easy
Bộ bé trai Animo
Bộ bé trai Animo Easy
Bộ chống muỗi set 2
Bộ nón bao tay chân
Bộ quần áo bé gái
Bộ quần áo bé trai
Bộ đông bé gái dài
Bộ đông bé trai dài
Hero lẻ 129k
Hộp quà
Khăn voan
Khẩu trang
Nón sơ sinh
Nón sơ sinh nhập khẩu
Phụ kiện cũ khác
Quần bé trai
Quần bé trai cũ
Quần dài bé trai
Quần legging bé gái
Quần lót bầu
Qu

In [134]:
import polars as pl

# Các category bạn phát hiện được
categories_to_check = [
    "0-12M Bodysuit bé trai đùi",
    "0-12M Bodysuit tam giác",
    "0-12M Bodysuit đông vải mỏng",
    "0-12M Bộ nón tay chân sơ sinh bo",
    "0-12M Quần sơ sinh ngắn",
    "0-12M Áo sơ sinh cài chéo tay ngắn",
    "0-24M Áo thun bé trai tay dài đông",
    "0-3Y Bộ đông bé trai dài",
]

# 1) Lọc subset Thời trang & age_group = "Không xác định"
fashion_unknown = df_filled.filter(
    (pl.col("category_l1") == "Thời trang")
    & (pl.col("age_group") == "Không xác định")
)

# 2) Tìm các dòng có category thuộc danh sách trên
matches = fashion_unknown.filter(
    pl.col("category").is_in(categories_to_check)
)

print("Số dòng match:", matches.height)
matches.head()


Số dòng match: 39


item_id,price,category_l1,category_l2,category_l3,category,description,brand,gender_target,age_group,item_type,description_new,gender_target_final
str,"decimal[38,4]",str,str,str,str,str,str,str,str,str,str,str
"""0024171040022""",19000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Phụ kiện khác""","""0-24M Áo thun bé trai tay dài đông""","""Set 3 quần lót bé trai Concung I047010 Xanh với ki…","""Con Cưng""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Bé Trai"""
"""3522000000140""",139000.0000,"""Thời trang""","""Quần áo & Phụ kiện sơ sinh""","""Áo""","""0-12M Áo sơ sinh cài chéo tay ngắn""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null,"""Sơ sinh"""
"""0909285860001""",19000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Phụ kiện khác""","""0-24M Áo thun bé trai tay dài đông""","""Không xác định""","""CF (ConCung Fashion)""","""Bé Trai""","""Không xác định""","""Đồ ngủ, đồ lót""","""Không xác định""","""Bé Trai"""
"""6048000000048""",229000.0000,"""Thời trang""","""Thời trang đông""","""Bodysuit đông""","""0-12M Bodysuit đông vải mỏng""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null,"""Không xác định"""
"""3496000000010""",75000.0000,"""Thời trang""","""Quần áo & Phụ kiện sơ sinh""","""Bao tay chân, nón""","""0-12M Bộ nón tay chân sơ sinh bo""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null,"""Sơ sinh"""


####
"0-12M Bodysuit bé trai đùi",
"0-12M Bodysuit tam giác",
"0-12M Bodysuit đông vải mỏng",
"0-12M Bộ nón tay chân sơ sinh bo",
"0-12M Quần sơ sinh ngắn",
"0-12M Áo sơ sinh cài chéo tay ngắn",
"0-24M Áo thun bé trai tay dài đông",
"0-3Y Bộ đông bé trai dài",

fill các `category` sau cho `age_group`

In [135]:
import polars as pl

# Giả sử df_filled là dataframe hiện tại (đã có gender_target_final, v.v.)
df_age = df_filled.with_columns(
    pl.col("age_group").alias("age_group_final")
)

mask_fashion_unknown_age = (
    (pl.col("category_l1") == "Thời trang")
    & (pl.col("age_group_final") == "Không xác định")
)

# Rule 1 – Fill cho các category bắt đầu bằng 0-12M
mask_012M = (
    mask_fashion_unknown_age
    & pl.col("category").str.starts_with("0-12M")
)

df_age = df_age.with_columns(
    pl.when(mask_012M)
      .then(pl.lit("0-12M"))
      .otherwise(pl.col("age_group_final"))
      .alias("age_group_final")
)

# Rule 2 – Fill cho các category bắt đầu bằng 0-24M
mask_024M = (
    mask_fashion_unknown_age
    & pl.col("category").str.starts_with("0-24M")
)

df_age = df_age.with_columns(
    pl.when(mask_024M)
      .then(pl.lit("0-24M"))
      .otherwise(pl.col("age_group_final"))
      .alias("age_group_final")
)

# Rule 3 – Fill cho các category bắt đầu bằng 0-3Y
mask_03Y = (
    mask_fashion_unknown_age
    & pl.col("category").str.starts_with("0-3Y")
)

df_age = df_age.with_columns(
    pl.when(mask_03Y)
      .then(pl.lit("0-3Y"))
      .otherwise(pl.col("age_group_final"))
      .alias("age_group_final")
)



In [136]:
filled_rows = df_age.filter(
    (pl.col("age_group") == "Không xác định")
    & (pl.col("age_group_final") != "Không xác định")
).height

print("Tổng số dòng được fill từ category:", filled_rows)


Tổng số dòng được fill từ category: 39


#### Lọc ra các class có trong mỗi `category`

In [137]:
import polars as pl

df = df_age   # hoặc df_filled tùy bạn đang dùng

# Hàm tiện dụng để lấy danh sách unique của 1 cột
def get_unique_list(df, col):
    return (
        df.select(col)
          .unique()
          .sort(col)
          .get_column(col)
          .to_list()
    )

# Lấy tất cả class
cat_l1_list = get_unique_list(df, "category_l1")
cat_l2_list = get_unique_list(df, "category_l2")
cat_l3_list = get_unique_list(df, "category_l3")
cat_list    = get_unique_list(df, "category")
age_list    = get_unique_list(df, "age_group")

# Gom vào dictionary
category_dict = {
    "category_l1": cat_l1_list,
    "category_l2": cat_l2_list,
    "category_l3": cat_l3_list,
    "category": cat_list,
    # "age_group": age_list
}

# In ra theo từng dòng, rất dễ đọc
print("\n=== DANH SÁCH TẤT CẢ CLASS CỦA TỪNG CATEGORY ===\n")
for key, value in category_dict.items():
    print(f"{key}: {value}\n")


=== DANH SÁCH TẤT CẢ CLASS CỦA TỪNG CATEGORY ===

category_l1: ['Babycare', 'Gói Hội Viên', 'Hóa mỹ phẩm cho bé', 'Hóa mỹ phẩm gia đình', 'Phụ kiện', 'Sữa', 'Sữa nước', 'TPCN', 'Textile', 'Thời trang', 'Thực phẩm cho bé', 'Thực phẩm cho gia đình', 'Tã', 'Vệ sinh', 'Đồ chơi & Sách']

category_l2: ['0-1Y', '1Y+', 'A2 milk company', 'Abbott', 'Alphagen', 'Animo', 'Anlene', 'Anmum', 'Appekidz', 'Aptamil', 'Aptamil Úc', 'Bellamy', 'Blackmores', 'Bobby', 'Bubs', 'Bánh', 'Bánh & Kẹo cho bé', 'Bé ngủ', 'Bình sữa, phụ kiện', 'Bông gạc', 'Bông tẩy trang', 'Băng vệ sinh', 'Bột lắc sữa', 'Bột ăn dặm', 'Caryn', 'Cho mẹ', 'Cháo ăn dặm', 'Chăm sóc cơ thể', 'Chăm sóc da', 'Chăm sóc da bé', 'Chăm sóc gia đình', 'Chăm sóc mẹ trước & sau sinh', 'Chăm sóc răng miệng', 'Chăm sóc sức khỏe bé', 'Chăm sóc tóc', 'Chăn', 'Chăn ga gối gia đình', 'Confidence', 'Cơ cấu hàng cũ', 'Cơ cấu hàng tồn', 'Dad and Me', 'Dầu sức khỏe', 'Dầu ăn & Gia vị', 'Elprairie', 'Enfa', 'FCV', 'Genki', 'Giày dép 1-3Y', 'Giày tập đi',

In [138]:
# Gom vào dictionary
category_dict = {
    "age_group": age_list
}

# In ra theo từng dòng, rất dễ đọc
print("\n=== DANH SÁCH TẤT CẢ CLASS CỦA age_group ===\n")
for key, value in category_dict.items():
    print(f"{key}: {value}\n")


=== DANH SÁCH TẤT CẢ CLASS CỦA age_group ===

age_group: ['0-10M', '0-12M', '0-12Y', '0-18M', '0-1M', '0-24M', '0-2M', '0-36M', '0-3M', '0-4Y', '0-5M', '0-5Y', '0-6M', '0-6Y', '0-9M', '10Y-12Y', '11Y-12Y', '12M-18M', '12M-4Y', '13M-24M', '18M-24M', '18M-36M', '18M-4Y', '1M-12M', '1M-15M', '1M-3M', '1Y-10Y', '1Y-11Y', '1Y-12Y', '1Y-16Y', '1Y-2Y', '1Y-3Y', '1Y-4Y', '1Y-6Y', '1Y-9Y', '2M-15M', '2M-6M', '2Y-10Y', '2Y-3Y', '2Y-6Y', '3M-12M', '3M-18M', '3M-24M', '3M-6M', '3Y-10Y', '3Y-12Y', '3Y-4Y', '3Y-5Y', '3Y-6Y', '4M-4Y', '4M-6M', '4Y-5Y', '5Y-6Y', '6M-10M', '6M-12M', '6M-12Y', '6M-15M', '6M-18M', '6M-24M', '6M-36M', '6M-5Y', '6M-6Y', '6M-9M', '7Y-10Y', '7Y-14Y', '7Y-8Y', '8Y-14Y', '9M-12M', '9M-15M', '9M-20M', '9M-24M', '9M-36M', '9M-4Y', '9M-6Y', '9Y-10Y', 'Không xác định', 'Mẹ', 'Trên 1Y', 'Trên 2Y', 'Trên 3Y', 'Trên 6M', 'Từ 0M', 'Từ 10M', 'Từ 10Y', 'Từ 12Y', 'Từ 13Y', 'Từ 18M', 'Từ 19M', 'Từ 19Y', 'Từ 1M', 'Từ 1Y', 'Từ 2M', 'Từ 2Y', 'Từ 3M', 'Từ 3Y', 'Từ 4M', 'Từ 4Y', 'Từ 5M', 'Từ 

#### Các `class` của mỗi `category` liên quan đến keyword "mẹ"

In [139]:
import polars as pl

df = df_age  # hoặc df_filled nếu bạn dùng dataframe khác

# === Danh sách bạn cung cấp ===

cat_l2_me = [
    "0-1Y",
    "1Y+",
    "Giày dép 1-3Y",
    "Cho mẹ",
    "Chăm sóc mẹ trước & sau sinh",
    "Gói Hội Viên BẦU",
    "Sữa cho mẹ",
    "TPCN cho mẹ",
    "Đồ dùng cho mẹ",
]

cat_l3_me = [
    "Dung dịch vệ sinh phụ nữ",
    "Gói Hội Viên BẦU",
    "Phụ kiện cho mẹ",
    "Vitamin mẹ bầu",
    "Đồ dùng cho mẹ ngừng bán",
]

cat_me = [
    "Balo, túi cho mẹ",
    "Bàn chải 0-1Y",
    "Bàn chải 1-2Y",
    "Bàn chải 2Y+",
    "Băng vệ sinh Diana sau sinh",
    "Colosbaby Mom",
    "Dung dịch vệ sinh phụ nữ Bimunica",
    "Dung dịch vệ sinh phụ nữ Chilly",
    "Dung dịch vệ sinh phụ nữ Crevil",
    "Dung dịch vệ sinh phụ nữ Felce Azzurra",
    "Dung dịch vệ sinh phụ nữ Fremfesh",
    "Dung dịch vệ sinh phụ nữ Lactacyd",
    "Dung dịch vệ sinh phụ nữ Lovely",
    "Dung dịch vệ sinh phụ nữ SAFORELLE",
    "Enfa Mom",
    "Gói Hội Viên Pink MOM All in",
    "Gói Hội Viên Pink MOM Easy",
    "Gói Hội Viên Pink MOM Upgrade",
    "Hi Mom",
    "Meiji Mom",
    "Miếng lót thấm sữa Aga-ae",
    "Miếng lót thấm sữa ChuchuBaby",
    "Miếng lót thấm sữa Pigeon",
    "Miếng lót thấm sữa ngừng bán",
    "Nịt bụng cho mẹ",
    "Phụ kiện cho mẹ ngừng bán",
    "Quần lót bầu",
    "Quần, áo, đầm bầu",
    "Similac Mom",
    "Vớ cho mẹ",
    "Wakodo Mom",
    "XO Mom",
    "Áo ngực cho mẹ",
]

In [140]:
print("\n=== category_l2: số lượng age_group = 'Không xác định' ===")

stats_l2 = (
    df.filter(
        (pl.col("category_l2").is_in(cat_l2_me))
        & (pl.col("age_group") == "Không xác định")
    )
    .group_by("category_l2")
    .agg(pl.len().alias("count_khong_xac_dinh"))
    .sort("count_khong_xac_dinh", descending=True)
)

print(stats_l2)


=== category_l2: số lượng age_group = 'Không xác định' ===
shape: (9, 2)
┌──────────────────────────────┬──────────────────────┐
│ category_l2                  ┆ count_khong_xac_dinh │
│ ---                          ┆ ---                  │
│ str                          ┆ u32                  │
╞══════════════════════════════╪══════════════════════╡
│ 1Y+                          ┆ 1138                 │
│ 0-1Y                         ┆ 415                  │
│ Giày dép 1-3Y                ┆ 333                  │
│ Đồ dùng cho mẹ               ┆ 54                   │
│ TPCN cho mẹ                  ┆ 44                   │
│ Chăm sóc mẹ trước & sau sinh ┆ 18                   │
│ Cho mẹ                       ┆ 3                    │
│ Gói Hội Viên BẦU             ┆ 3                    │
│ Sữa cho mẹ                   ┆ 1                    │
└──────────────────────────────┴──────────────────────┘


In [141]:
print("\n=== category_l3: số lượng age_group = 'Không xác định' ===")

stats_l3 = (
    df.filter(
        (pl.col("category_l3").is_in(cat_l3_me))
        & (pl.col("age_group") == "Không xác định")
    )
    .group_by("category_l3")
    .agg(pl.len().alias("count_khong_xac_dinh"))
    .sort("count_khong_xac_dinh", descending=True)
)

print(stats_l3)


=== category_l3: số lượng age_group = 'Không xác định' ===
shape: (5, 2)
┌──────────────────────────┬──────────────────────┐
│ category_l3              ┆ count_khong_xac_dinh │
│ ---                      ┆ ---                  │
│ str                      ┆ u32                  │
╞══════════════════════════╪══════════════════════╡
│ Đồ dùng cho mẹ ngừng bán ┆ 26                   │
│ Dung dịch vệ sinh phụ nữ ┆ 12                   │
│ Vitamin mẹ bầu           ┆ 10                   │
│ Phụ kiện cho mẹ          ┆ 3                    │
│ Gói Hội Viên BẦU         ┆ 3                    │
└──────────────────────────┴──────────────────────┘


In [142]:
print("\n=== category (tên chi tiết): số lượng age_group = 'Không xác định' ===")

stats_cat = (
    df.filter(
        (pl.col("category").is_in(cat_me))
        & (pl.col("age_group") == "Không xác định")
    )
    .group_by("category")
    .agg(pl.len().alias("count_khong_xac_dinh"))
    .sort("count_khong_xac_dinh", descending=True)
)

print(stats_cat)



=== category (tên chi tiết): số lượng age_group = 'Không xác định' ===
shape: (29, 2)
┌────────────────────────────────────────┬──────────────────────┐
│ category                               ┆ count_khong_xac_dinh │
│ ---                                    ┆ ---                  │
│ str                                    ┆ u32                  │
╞════════════════════════════════════════╪══════════════════════╡
│ Quần lót bầu                           ┆ 29                   │
│ Bàn chải 2Y+                           ┆ 28                   │
│ Balo, túi cho mẹ                       ┆ 15                   │
│ Bàn chải 0-1Y                          ┆ 10                   │
│ Enfa Mom                               ┆ 8                    │
│ Bàn chải 1-2Y                          ┆ 8                    │
│ Nịt bụng cho mẹ                        ┆ 6                    │
│ Dung dịch vệ sinh phụ nữ Lactacyd      ┆ 4                    │
│ Quần, áo, đầm bầu                      ┆ 4           

In [143]:
import polars as pl

df = df_age   # hoặc df_filled nếu bạn dùng dataframe khác

cat_l2_me = [
    "0-1Y",
    "1Y+",
    "Giày dép 1-3Y",
    "Cho mẹ",
    "Chăm sóc mẹ trước & sau sinh",
    "Gói Hội Viên BẦU",
    "Sữa cho mẹ",
    "TPCN cho mẹ",
    "Đồ dùng cho mẹ",
]

cat_l3_me = [
    "Dung dịch vệ sinh phụ nữ",
    "Gói Hội Viên BẦU",
    "Phụ kiện cho mẹ",
    "Vitamin mẹ bầu",
    "Đồ dùng cho mẹ ngừng bán",
]

cat_me = [
    "Balo, túi cho mẹ",
    "Bàn chải 0-1Y",
    "Bàn chải 1-2Y",
    "Bàn chải 2Y+",
    "Băng vệ sinh Diana sau sinh",
    "Colosbaby Mom",
    "Dung dịch vệ sinh phụ nữ Bimunica",
    "Dung dịch vệ sinh phụ nữ Chilly",
    "Dung dịch vệ sinh phụ nữ Crevil",
    "Dung dịch vệ sinh phụ nữ Felce Azzurra",
    "Dung dịch vệ sinh phụ nữ Fremfesh",
    "Dung dịch vệ sinh phụ nữ Lactacyd",
    "Dung dịch vệ sinh phụ nữ Lovely",
    "Dung dịch vệ sinh phụ nữ SAFORELLE",
    "Enfa Mom",
    "Gói Hội Viên Pink MOM All in",
    "Gói Hội Viên Pink MOM Easy",
    "Gói Hội Viên Pink MOM Upgrade",
    "Hi Mom",
    "Meiji Mom",
    "Miếng lót thấm sữa Aga-ae",
    "Miếng lót thấm sữa ChuchuBaby",
    "Miếng lót thấm sữa Pigeon",
    "Miếng lót thấm sữa ngừng bán",
    "Nịt bụng cho mẹ",
    "Phụ kiện cho mẹ ngừng bán",
    "Quần lót bầu",
    "Quần, áo, đầm bầu",
    "Similac Mom",
    "Vớ cho mẹ",
    "Wakodo Mom",
    "XO Mom",
    "Áo ngực cho mẹ",
]

# =============================
# TẠO DATAFRAME GỘP TẤT CẢ CLASS MẸ CÓ age_group = "Không xác định"
# =============================
df_me_unknown = (
    df.filter(
        (pl.col("age_group") == "Không xác định")
        & (
            pl.col("category_l2").is_in(cat_l2_me)
            | pl.col("category_l3").is_in(cat_l3_me)
            | pl.col("category").is_in(cat_me)
        )
    )
)

print(df_me_unknown)
print("\nSố dòng:", df_me_unknown.height)


shape: (2_116, 14)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ item_id   ┆ price     ┆ category_ ┆ category_ ┆ … ┆ item_type ┆ descripti ┆ gender_ta ┆ age_grou │
│ ---       ┆ ---       ┆ l1        ┆ l2        ┆   ┆ ---       ┆ on_new    ┆ rget_fina ┆ p_final  │
│ str       ┆ decimal[3 ┆ ---       ┆ ---       ┆   ┆ str       ┆ ---       ┆ l         ┆ ---      │
│           ┆ 8,4]      ┆ str       ┆ str       ┆   ┆           ┆ str       ┆ ---       ┆ str      │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆ str       ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 000801000 ┆ 45000.000 ┆ Đồ chơi & ┆ 0-1Y      ┆ … ┆ Không xác ┆ Chi tiết  ┆ Không xác ┆ Không    │
│ 0015      ┆ 0         ┆ Sách      ┆           ┆   ┆ định      ┆ sản phẩm  ┆ định      ┆ xác định │
│           ┆           ┆           ┆           ┆   ┆           ┆ Tên sả

#### Fill các `class` của mỗi `category` trên vào cột `age_group`
-> Ngoài ra còn nhận thấy cột `gender_target` cũng có nhiều giá trị "không xác định" tại các giá trị class này -> đề xuất thêm một `gender_target` là `Phụ Nữ`

Fill giá trị "Mẹ"
Rule:

- Fill age_group_final = "Mẹ" nếu:

    1. age_group_final hiện tại = "Không xác định" và

    2. Một trong các điều kiện sau:

        - category ∈
        "Băng vệ sinh Abena sau sinh",
        "Băng vệ sinh Goodfeel"

        - HOẶC category == "Băng vệ sinh khác" và description chứa "Laurier Happy Skin".

In [144]:
import polars as pl

# ==========================================
# 0) KHỞI TẠO age_group_final
# ==========================================
df_age = df_filled.with_columns(
    pl.col("age_group").alias("age_group_final")
)

# ==========================================
# 1) THỐNG KÊ TRƯỚC KHI FILL
# ==========================================
total_rows = df_age.height

unknown_before = df_age.filter(
    pl.col("age_group_final") == "Không xác định"
).height

ratio_before = unknown_before / total_rows * 100

print("=== TRƯỚC KHI FILL AGE_GROUP ===")
print(f"Số dòng 'Không xác định': {unknown_before:,} / {total_rows:,} "
      f"({ratio_before:.2f}%)")

# ==========================================
# 2) KHAI BÁO CÁC CLASS LIÊN QUAN ĐẾN MẸ
# ==========================================

cat_l2_me = [
    "0-1Y",
    "1Y+",
    "Giày dép 1-3Y",
    "Cho mẹ",
    "Chăm sóc mẹ trước & sau sinh",
    "Gói Hội Viên BẦU",
    "Sữa cho mẹ",
    "TPCN cho mẹ",
    "Đồ dùng cho mẹ",
]

cat_l3_me = [
    "Dung dịch vệ sinh phụ nữ",
    "Gói Hội Viên BẦU",
    "Phụ kiện cho mẹ",
    "Vitamin mẹ bầu",
    "Đồ dùng cho mẹ ngừng bán",
]

cat_me = [
    "Balo, túi cho mẹ",
    "Bàn chải 0-1Y",
    "Bàn chải 1-2Y",
    "Bàn chải 2Y+",
    "Băng vệ sinh Diana sau sinh",
    "Băng vệ sinh Abena sau sinh",
    "Băng vệ sinh Goodfeel",
    "Colosbaby Mom",
    "Dung dịch vệ sinh phụ nữ Bimunica",
    "Dung dịch vệ sinh phụ nữ Chilly",
    "Dung dịch vệ sinh phụ nữ Crevil",
    "Dung dịch vệ sinh phụ nữ Felce Azzurra",
    "Dung dịch vệ sinh phụ nữ Fremfesh",
    "Dung dịch vệ sinh phụ nữ Lactacyd",
    "Dung dịch vệ sinh phụ nữ Lovely",
    "Dung dịch vệ sinh phụ nữ SAFORELLE",
    "Enfa Mom",
    "Gói Hội Viên Pink MOM All in",
    "Gói Hội Viên Pink MOM Easy",
    "Gói Hội Viên Pink MOM Upgrade",
    "Hi Mom",
    "Meiji Mom",
    "Miếng lót thấm sữa Aga-ae",
    "Miếng lót thấm sữa ChuchuBaby",
    "Miếng lót thấm sữa Pigeon",
    "Miếng lót thấm sữa ngừng bán",
    "Nịt bụng cho mẹ",
    "Phụ kiện cho mẹ ngừng bán",
    "Quần lót bầu",
    "Quần, áo, đầm bầu",
    "Similac Mom",
    "Vớ cho mẹ",
    "Wakodo Mom",
    "XO Mom",
    "Áo ngực cho mẹ",
]

# Chỉ fill cho các dòng hiện đang "Không xác định"
base_mask_unknown = pl.col("age_group_final") == "Không xác định"

# ==========================================
# 3) FILL CÁC CLASS CÓ MỐC TUỔI RÕ RÀNG
#    3.1. Từ category (Bàn chải ...)
# ==========================================

mask_cat_brush_01 = base_mask_unknown & (pl.col("category") == "Bàn chải 0-1Y")
mask_cat_brush_12 = base_mask_unknown & (pl.col("category") == "Bàn chải 1-2Y")
mask_cat_brush_2p = base_mask_unknown & (pl.col("category") == "Bàn chải 2Y+")

df_age = df_age.with_columns(
    pl.when(mask_cat_brush_01)
      .then(pl.lit("0-1Y"))
      .when(mask_cat_brush_12)
      .then(pl.lit("1-2Y"))
      .when(mask_cat_brush_2p)
      .then(pl.lit("2Y+"))
      .otherwise(pl.col("age_group_final"))
      .alias("age_group_final")
)

# ==========================================
#    3.2. Từ category_l2 (0-1Y, 1Y+, Giày dép 1-3Y)
# ==========================================

# Cập nhật lại base_mask_unknown sau lần fill trên
base_mask_unknown = pl.col("age_group_final") == "Không xác định"

mask_l2_01 = base_mask_unknown & (pl.col("category_l2") == "0-1Y")
mask_l2_1p = base_mask_unknown & (pl.col("category_l2") == "1Y+")
mask_l2_shoes_13 = base_mask_unknown & (pl.col("category_l2") == "Giày dép 1-3Y")

df_age = df_age.with_columns(
    pl.when(mask_l2_01)
      .then(pl.lit("0-1Y"))
      .when(mask_l2_1p)
      .then(pl.lit("Từ 1Y"))   # theo yêu cầu: 1Y+ → "Từ 1Y"
      .when(mask_l2_shoes_13)
      .then(pl.lit("1-3Y"))
      .otherwise(pl.col("age_group_final"))
      .alias("age_group_final")
)

# ==========================================
# 4) FILL CÁC CLASS LIÊN QUAN ĐẾN "MẸ"
#    (ngoại trừ các class tuổi đã fill vì giờ chúng không còn "Không xác định")
# ==========================================

base_mask_unknown = pl.col("age_group_final") == "Không xác định"

mask_laurier = (
    base_mask_unknown
    & (pl.col("category") == "Băng vệ sinh khác")
    & pl.col("description").str.contains("Laurier Happy Skin", literal=False)
)

mask_me = base_mask_unknown & (
    pl.col("category_l2").is_in(cat_l2_me)
    | pl.col("category_l3").is_in(cat_l3_me)
    | pl.col("category").is_in(cat_me)
)

mask_me_all = mask_me | mask_laurier

df_age = df_age.with_columns(
    pl.when(mask_me_all)
      .then(pl.lit("Mẹ"))
      .otherwise(pl.col("age_group_final"))
      .alias("age_group_final")
)

# ==========================================
# 5) THỐNG KÊ SAU KHI FILL
# ==========================================

unknown_after = df_age.filter(
    pl.col("age_group_final") == "Không xác định"
).height

ratio_after = unknown_after / total_rows * 100

print("\n=== SAU KHI FILL AGE_GROUP (TUỔI + MẸ) ===")
print(f"Số dòng 'Không xác định': {unknown_after:,} / {total_rows:,} "
      f"({ratio_after:.2f}%)")

# Nếu muốn xem nhanh các dòng đã được fill từ "Không xác định" → khác:
df_filled_age_changes = df_age.filter(
    (pl.col("age_group") == "Không xác định")
    & (pl.col("age_group_final") != "Không xác định")
)

print("\nSố dòng được fill từ 'Không xác định' → giá trị khác:",
      df_filled_age_changes.height)
print("\n5 dòng mẫu đã được fill:")
print(df_filled_age_changes.head(5))


=== TRƯỚC KHI FILL AGE_GROUP ===
Số dòng 'Không xác định': 15,803 / 27,323 (57.84%)

=== SAU KHI FILL AGE_GROUP (TUỔI + MẸ) ===
Số dòng 'Không xác định': 13,681 / 27,323 (50.07%)

Số dòng được fill từ 'Không xác định' → giá trị khác: 2122

5 dòng mẫu đã được fill:
shape: (5, 14)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ item_id   ┆ price     ┆ category_ ┆ category_ ┆ … ┆ item_type ┆ descripti ┆ gender_ta ┆ age_grou │
│ ---       ┆ ---       ┆ l1        ┆ l2        ┆   ┆ ---       ┆ on_new    ┆ rget_fina ┆ p_final  │
│ str       ┆ decimal[3 ┆ ---       ┆ ---       ┆   ┆ str       ┆ ---       ┆ l         ┆ ---      │
│           ┆ 8,4]      ┆ str       ┆ str       ┆   ┆           ┆ str       ┆ ---       ┆ str      │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆ str       ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 000801000 ┆

#### Lọc tất cả các dòng có category chứa cụm “Băng vệ sinh” để soi và tiếp tục thiết kế rule fill.

In [145]:
import polars as pl

# 1) Lọc tất cả dòng có 'Băng vệ sinh' trong category
df_bangveSinh = (
    df_age
    .filter(
        pl.col("category").str.contains("Băng vệ sinh", literal=False)
    )
)

print("\n=== TẤT CẢ DÒNG CÓ 'Băng vệ sinh' TRONG category ===")
print("\nSố dòng:", df_bangveSinh.height)
df_bangveSinh.head()    



=== TẤT CẢ DÒNG CÓ 'Băng vệ sinh' TRONG category ===

Số dòng: 32


item_id,price,category_l1,category_l2,category_l3,category,description,brand,gender_target,age_group,item_type,description_new,gender_target_final,age_group_final
str,"decimal[38,4]",str,str,str,str,str,str,str,str,str,str,str,str
"""1782000000001""",89000.0000,"""Vệ sinh""","""Băng vệ sinh""","""Băng vệ sinh sau sinh""","""Băng vệ sinh Abena sau sinh""","""Sau khi sinh, tùy thuộc vào cơ địa từng người mà s…","""Abena""","""Không xác định""","""Không xác định""","""Băng vệ sinh""","""Chi tiết sản phẩm Tên sản phẩm…","""Không xác định""","""Mẹ"""
"""1944000000133""",115000.0000,"""Vệ sinh""","""Băng vệ sinh""","""Băng vệ sinh thường""","""Băng vệ sinh khác""","""Băng vệ sinh (BVS) Laurier Happy Skin là nhóm sản …","""Laurier""","""Không xác định""","""Không xác định""","""Băng vệ sinh""","""Chi tiết sản phẩm Tên sản phẩm…","""Không xác định""","""Mẹ"""
"""7149000000001""",89000.0000,"""Vệ sinh""","""Băng vệ sinh""","""Băng vệ sinh thường""","""Băng vệ sinh Goodfeel""","""﻿﻿﻿Băng vệ sinh Goodfeel là sản phẩm dành cho mẹ c…","""Goodfeel""","""Không xác định""","""Không xác định""","""Băng vệ sinh""","""Chi tiết sản phẩm Tên sản phẩm…","""Không xác định""","""Mẹ"""
"""0007190000004""",21000.0000,"""Vệ sinh""","""Băng vệ sinh""","""Băng vệ sinh thường""","""Băng vệ sinh Diana thường""","""Băng vệ sinh Diana Sensi Cool Fresh 23cm (8 miếng)…","""Diana""","""Không xác định""","""Không xác định""","""Băng vệ sinh""","""Chi tiết sản phẩm Tên sản phẩm…","""Không xác định""","""Không xác định"""
"""3269000000001""",92000.0000,"""Vệ sinh""","""Băng vệ sinh""","""Băng vệ sinh thường""","""Băng vệ sinh Diana thường""","""﻿Ưu điểm nổi bật . 360 ĐỘ CHỐNG TRÀN: Thiết kế độc…","""Diana""","""Không xác định""","""Không xác định""","""Băng vệ sinh""","""Chi tiết sản phẩm Tên sản phẩm…","""Không xác định""","""Không xác định"""


Không thể fill được nữa

In [146]:
# import polars as pl

# # Lọc các dòng có 'Băng vệ sinh' trong category
# df_bvs = (
#     df_age
#     .filter(
#         pl.col("category").str.contains("Băng vệ sinh", literal=False)
#     )
#     .select([
#         "category",
#         "description",
#         "description_new",
#         "age_group_final",
#     ])
#     .with_row_count("idx")   # tạo index 0,1,2,...
# )

# # In chỉ dòng đầu tiên
# if df_bvs.height > 0:
#     row = df_bvs.row(17, named=True)   # lấy dòng đầu tiên
#     print("\n===== DÒNG ĐẦU TIÊN =====")
#     print("idx             :", row["idx"])
#     print("category        :", row["category"])
#     print("description     :", row["description"])
#     print("description_new :", row["description_new"])
#     print("age_group_final :", row["age_group_final"])
# else:
#     print("Không có dòng nào chứa 'Băng vệ sinh' trong category.")


In [147]:
DATA_PATH = r".././dataset/sales_pers.item_chunk_0.parquet"

df_raw = pl.read_parquet(DATA_PATH)

In [148]:
# a = df.filter(pl.col("brand") == "Chupa Chups")
a = df_raw.filter(pl.col("category").str.to_lowercase() == "la vie")
a

p_id,item_id,price,category_l1_id,category_l1,category_l2_id,category_l2,category_l3_id,category_l3,category_id,category,description,brand,manufacturer,creation_timestamp,is_deleted,created_date,updated_date,sync_status_id,last_sync_date,sync_error_message,image_url,gender_target,age_group,item_type,gp,weight,color,size,origin,volume,material,sale_status,description_new
i32,str,"decimal[38,4]",i32,str,i32,str,i32,str,i32,str,str,str,str,i64,bool,datetime[μs],datetime[μs],i32,datetime[μs],str,str,str,str,str,"decimal[38,4]",f32,str,str,str,str,str,i32,str
108796,"""5114000000002""",8000.0000,5126,"""Thực phẩm cho gia đình""",5284,"""Đồ uống""",5287,"""Nước suối""",5679,"""La Vie""","""Không xác định""","""La Vie""","""Không xác định""",1641577274,false,2022-01-07 17:41:14.487,2025-08-18 09:59:19.847,2,2025-07-18 17:59:29.898256,null,"""Không xác định""","""Không xác định""","""Không xác định""","""Thức uống dinh dưỡng""",3024.0000,null,"""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",0,"""Chi tiết sản phẩm Tên sản phẩm…"
108798,"""5114000000004""",12000.0000,5126,"""Thực phẩm cho gia đình""",5284,"""Đồ uống""",5287,"""Nước suối""",5679,"""La Vie""","""Không xác định""","""La Vie""","""Không xác định""",1641577306,false,2022-01-07 17:41:46.587,2025-08-18 09:59:19.847,2,2025-07-18 17:59:29.898256,null,"""Không xác định""","""Không xác định""","""Không xác định""","""Thức uống dinh dưỡng""",4536.0000,null,"""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",0,"""Chi tiết sản phẩm Tên sản phẩm…"
108797,"""5114000000003""",10000.0000,5126,"""Thực phẩm cho gia đình""",5284,"""Đồ uống""",5287,"""Nước suối""",5679,"""La Vie""","""Với nguồn nước tinh khiết vị thiên nhiên trong làn…","""La Vie""","""Không xác định""",1641577293,false,2022-01-07 17:41:33.207,2025-09-27 00:05:36.233,2,2025-07-18 17:59:29.898256,null,"""Không xác định""","""Không xác định""","""Không xác định""","""Thức uống dinh dưỡng""",3780.0000,null,"""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",0,"""Chi tiết sản phẩm Tên sản phẩm…"
108795,"""5114000000001""",6000.0000,5126,"""Thực phẩm cho gia đình""",5284,"""Đồ uống""",5287,"""Nước suối""",5679,"""La Vie""","""﻿Nước khoáng Lavie được sản xuất từ 100% nước khoá…","""La Vie""","""Không xác định""",1641577249,false,2022-01-07 17:40:49.487,2025-09-26 08:05:02.743,2,2025-07-18 17:59:29.898256,null,"""Không xác định""","""Không xác định""","""Không xác định""","""Thức uống dinh dưỡng""",2268.0000,null,"""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",1,"""Chi tiết sản phẩm Tên sản phẩm: Nước…"
136079,"""5679000000001""",6000.0000,5126,"""Thực phẩm cho gia đình""",5284,"""Đồ uống""",5287,"""Nước suối""",5679,"""La Vie""","""Không xác định""","""Không xác định""","""Không xác định""",1718106122,false,2024-06-11 11:42:02.770,2025-07-07 15:33:37.855222,2,2025-07-18 17:59:29.898256,null,"""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",2268.0000,null,"""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",1,null
124828,"""4124000000011""",5000.0000,5126,"""Thực phẩm cho gia đình""",5284,"""Đồ uống""",5287,"""Nước suối""",5679,"""La Vie""","""Không xác định""","""Không xác định""","""Không xác định""",1686240264,false,2023-06-08 16:04:24.050,2025-07-07 15:33:37.855222,2,2025-07-18 17:59:29.898256,null,"""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",1890.0000,null,"""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",1,null


#### Ý tưởng 1 – Dùng regex bắt tuổi/tháng ghi rõ trong mô tả (độ chắc chắn rất cao)

Thống kê

In [149]:
import re

# ==============================
# 1. Cấu hình context & từ khóa
# ==============================

import re

# ==============================
# 1. Cấu hình context & từ khóa
# ==============================

POS_CONTEXT = [
    "bé", "trẻ", "trẻ em", "em bé",
    "baby", "kid", "child", "children",
    "thiếu nhi", "newborn", "sơ sinh",
    "tháng tuổi", "months old", "years old",
]

CHILD_WORDS = [
    "bé", "trẻ", "trẻ em", "em bé",
    "baby", "kid", "child", "children",
]

NEG_STRONG = [
    # hạn sử dụng / bảo quản
    "sử dụng trong vòng",
    "hạn sử dụng", "thời hạn", "hạn dùng", "hạn sử dụng tốt nhất",
    "bảo quản", "kể từ ngày sản xuất", "sau khi mở nắp",

    # không dùng / không phù hợp cho trẻ
    "không thích hợp cho trẻ",
    "không thích hợp với trẻ",
    "không dùng cho trẻ",
    "không nên dùng cho trẻ",
    "không phù hợp cho trẻ",
    "không phù hợp với trẻ",
    "không sử dụng sản phẩm cho trẻ",
    "không sử dụng cho trẻ",

    # tiếng Anh
    "not suitable for children",
    "do not use for children",
]

# Từ khóa liên quan tới cân nặng (để loại khỏi tuổi)
WEIGHT_WORDS = [
    "kg", "kilogram", "ký", "kí", "cân", "nặng", "trọng lượng", "weight"
]

MONTH_WORDS = ["tháng", "thang", "month", "months"]
YEAR_WORDS  = ["tuổi", "year", "years"]
WEEK_WORDS  = ["tuần", "tuan", "week", "weeks"]

WINDOW_POS = 20   # window cho POS context
WINDOW_NEG = 80  # window cho NEG context


# ==============================
# 2. Regex patterns cho EXTRACT
# ==============================

# 2.1. Range kiểu "0-6 tháng ... đến ... 3 tuổi"
CROSS_MONTH_RANGE_TO_YEAR_PATTERN = re.compile(
    r"(?P<m_start>\d{1,2})\s*-\s*(?P<m_end>\d{1,2})\s*"
    r"(?:tháng|thang|month|months)"
    r".{0,40}?(?:đến|tới|to)\s*.{0,40}?"
    r"(?P<y_end>\d{1,2}(?:[.,]\d+)?)\s*(?:tuổi|year|years)",
)

# 2.2. Newborn range "sơ sinh ... đến ... 3 tuổi"
NEWBORN_RANGE_PATTERN = re.compile(
    r"(?:từ\s+giai\s*đoạn\s+)?sơ\s*sinh"
    r".{0,30}?(?:đến|tới|to|-).{0,30}?"
    r"\d{1,2}(?:[.,]\d+)?\s*(tháng|thang|month|months|tuổi|year|years)",
)

# 2.3. "độ tuổi từ 2 đến 6" (có thể có chữ 'phù hợp', ':', ...)
AGE_WORD_RANGE_PATTERN = re.compile(
    r"độ\s*tuổi[^0-9]{0,20}"      # 'độ tuổi', 'độ tuổi phù hợp:'
    r"(?:từ\s*)?"
    r"(?P<n1>\d{1,2}(?:[.,]\d+)?)\s*(?:-|–|—|đến|to)\s*"
    r"(?P<n2>\d{1,2}(?:[.,]\d+)?)"
)

# 2.4. Range 2 đơn vị: "6 tháng đến 5 tuổi", "1 tuần - 1 tuổi"
RANGE_BOTH_UNITS_PATTERN = re.compile(
    r"(?P<n1>\d{1,2}(?:[.,]\d+)?)\s*"
    r"(?P<u1>tháng tuổi|tháng|thang|month|months|tuổi|year|years|tuần|tuan|week|weeks)\s*"
    r"(?:-|–|—|đến|to)\s*"
    r"(?P<n2>\d{1,2}(?:[.,]\d+)?)\s*"
    r"(?P<u2>tháng tuổi|tháng|thang|month|months|tuổi|year|years|tuần|tuan|week|weeks)"
)

# 2.5. Range 1 đơn vị: "0-12 tháng", "2-6 tuổi", "2-4 tuần"
RANGE_ONE_UNIT_PATTERN = re.compile(
    r"(?:(?:từ)\s*)?"
    r"(?P<n1>\d{1,2}(?:[.,]\d+)?)\s*"
    r"(?:-|–|—|đến|to)\s*"
    r"(?P<n2>\d{1,2}(?:[.,]\d+)?)\s*"
    r"(?P<u>tháng tuổi|tháng|thang|month|months|tuổi|year|years|tuần|tuan|week|weeks)"
)

# 2.6. Single: "6 tháng", "từ 6 tháng", "trên 1 tuổi", "dưới 12 tháng",
#              "6 tháng tuổi trở lên", "sau 6 tháng tuổi", "từ 2.5 tuổi", ...
SINGLE_PATTERN = re.compile(
    r"(?:(?P<cmp>dưới|under|<|từ|trên|hơn|sau|>=|over|more than)\s*)?"
    r"(?P<n>\d{1,2}(?:[.,]\d+)?)\s*"
    r"(?P<u>tháng tuổi|tháng|thang|month|months|tuổi|year|years|tuần|tuan|week|weeks)"
    r"(?:\s*(?P<suffix>trở lên|\+))?"
)



# ==============================
# 3. Hàm context POS/NEG
# ==============================

def has_pos_context(context: str) -> bool:
    return any(tok in context for tok in POS_CONTEXT)


def has_neg_context(context: str) -> bool:
    ctx = context

    # 1) Các cụm NEG mạnh (hạn sử dụng, bảo quản, ...)
    if any(phrase in ctx for phrase in NEG_STRONG):
        return True

    # 2) "không ... (phù hợp/thích hợp/dùng/sử dụng) ... trẻ"
    if "không" in ctx and any(v in ctx for v in ["phù hợp", "thích hợp", "dùng", "sử dụng"]):
        if any(c in ctx for c in CHILD_WORDS):
            return True

    # 3) "tránh dùng / tránh sử dụng ... cho trẻ"
    if any(start in ctx for start in ["tránh dùng", "tránh sử dụng"]):
        if any(c in ctx for c in CHILD_WORDS):
            return True

    # 4) English NEG
    if "not suitable" in ctx and any(c in ctx for c in ["child", "children", "kid", "baby"]):
        return True
    if "do not use" in ctx and any(c in ctx for c in ["child", "children", "kid", "baby"]):
        return True

    # 5) CASE THỜI LƯỢNG "6 tháng đầu", "trong 6 tháng đầu", "giai đoạn 6 tháng đầu"
    if "tháng đầu" in ctx or "tháng đầu đời" in ctx:
        if any(k in ctx for k in ["trong ", "trong vòng", "giai đoạn"]):
            return True

    # 6) CASE CÂN NẶNG: có "nặng", "kg", "cân", ... -> không coi là tuổi
    if any(w in ctx for w in WEIGHT_WORDS):
        return True

    return False


# ==============================
# 4. Hàm extract_age_phrases
# ==============================

def extract_age_phrases(desc: str) -> list[str]:
    """
    Trả về list các cụm tuổi/tháng trong desc, gồm:
      - range: '0-3 tháng', '3-6 tháng tuổi', '6 đến 11 tháng',
               '1-3 tuổi', '6 tháng đến 5 tuổi',
               '0-6 tháng đến bé 3 tuổi',
               'sơ sinh đến 3 tuổi', 'độ tuổi từ 2 đến 6',
               'trẻ từ 1 tuần - 1 tuổi', ...
      - single: 'dưới 12 tháng', 'từ 1 tuổi', 'trên 1 tuổi',
                '6 tháng tuổi trở lên', 'sau 6 tháng tuổi',
                'từ 2.5 tuổi', 'từ 3.5 tuổi', ...

    Rule:
      - CHỈ nhìn phần TRƯỚC "Lưu ý".
      - Phải có POS context (bé/trẻ/...) trong WINDOW_POS.
      - Nếu trong WINDOW_NEG có NEG context ('không phù hợp/không sử dụng/...'
        hoặc '6 tháng đầu', 'trong 6 tháng đầu', 'trẻ nặng ...', ...) thì BỎ.
      - Với single, nếu trái có 'độ tuổi phù hợp' thì coi là min age -> ép 'từ ...'.
    """
    if desc is None:
        return []

    full_text = desc.lower()

    # Cắt bỏ phần sau 'lưu ý'
    m_luuy = re.search(r"lưu\s*ý\b", full_text)
    if m_luuy:
        text = full_text[:m_luuy.start()]
    else:
        text = full_text

    matches: list[str] = []
    used_spans: list[tuple[int, int]] = []

    def overlap_span(start: int, end: int) -> bool:
        return any(not (end <= s or start >= e) for (s, e) in used_spans)

    def get_contexts(start_idx: int, end_idx: int) -> tuple[str, str]:
        left_pos = max(0, start_idx - WINDOW_POS)
        right_pos = min(len(text), end_idx + WINDOW_POS)
        context_pos = text[left_pos:right_pos]

        left_neg = max(0, start_idx - WINDOW_NEG)
        right_neg = min(len(text), end_idx + WINDOW_NEG)
        context_neg = text[left_neg:right_neg]

        return context_pos, context_neg

    # 0) Range kiểu "0-6 tháng ... đến ... 3 tuổi"
    for m in CROSS_MONTH_RANGE_TO_YEAR_PATTERN.finditer(text):
        start_idx, end_idx = m.span()
        context_pos, context_neg = get_contexts(start_idx, end_idx)
        if not has_pos_context(context_pos) or has_neg_context(context_neg):
            continue
        matches.append(text[start_idx:end_idx].strip())
        used_spans.append((start_idx, end_idx))

    # 1) Range newborn "sơ sinh ... đến ... 3 tuổi"
    for m in NEWBORN_RANGE_PATTERN.finditer(text):
        start_idx, end_idx = m.span()
        if overlap_span(start_idx, end_idx):
            continue
        context_pos, context_neg = get_contexts(start_idx, end_idx)
        if not has_pos_context(context_pos) or has_neg_context(context_neg):
            continue
        matches.append(text[start_idx:end_idx].strip())
        used_spans.append((start_idx, end_idx))

    # 2) Range 2 đơn vị: '6 tháng đến 5 tuổi', 'từ 1 tuần - 1 tuổi'
    for m in RANGE_BOTH_UNITS_PATTERN.finditer(text):
        start_idx, end_idx = m.span()
        if overlap_span(start_idx, end_idx):
            continue
        context_pos, context_neg = get_contexts(start_idx, end_idx)
        if not has_pos_context(context_pos) or has_neg_context(context_neg):
            continue
        matches.append(text[start_idx:end_idx].strip())
        used_spans.append((start_idx, end_idx))

    # 3) Range 1 đơn vị: '0-12 tháng', '2-6 tuổi', '1-4 tuần'
    for m in RANGE_ONE_UNIT_PATTERN.finditer(text):
        start_idx, end_idx = m.span()
        if overlap_span(start_idx, end_idx):
            continue
        context_pos, context_neg = get_contexts(start_idx, end_idx)
        if not has_pos_context(context_pos) or has_neg_context(context_neg):
            continue
        matches.append(text[start_idx:end_idx].strip())
        used_spans.append((start_idx, end_idx))

    # 4) "độ tuổi từ 2 đến 6"
    for m in AGE_WORD_RANGE_PATTERN.finditer(text):
        start_idx, end_idx = m.span()
        if overlap_span(start_idx, end_idx):
            continue
        context_pos, context_neg = get_contexts(start_idx, end_idx)
        if not has_pos_context(context_pos) or has_neg_context(context_neg):
            continue
        matches.append(text[start_idx:end_idx].strip())
        used_spans.append((start_idx, end_idx))

    # 5) Single: "dưới 12 tháng", "từ 6 tháng", "6 tháng tuổi trở lên",
    #            "sau 6 tháng tuổi", "từ 2.5 tuổi", ...
    for m in SINGLE_PATTERN.finditer(text):
        start_idx, end_idx = m.span()
        if overlap_span(start_idx, end_idx):
            continue

        context_pos, context_neg = get_contexts(start_idx, end_idx)
        if not has_pos_context(context_pos) or has_neg_context(context_neg):
            continue

        match_text = text[start_idx:end_idx].strip()

        # Nếu trước cụm này có 'độ tuổi phù hợp' ... => ép thành "từ ..."
        ctx_left = text[max(0, start_idx - 60):start_idx]
        if any(kw in ctx_left for kw in [
            "độ tuổi phù hợp", "độ tuổi sử dụng", "độ tuổi khuyến nghị", "độ tuổi khuyên dùng"
        ]):
            if not re.search(r"\b(từ|trên|hơn|sau|dưới|under|over|more than)\b", match_text):
                match_text = "từ " + match_text

        matches.append(match_text)
        used_spans.append((start_idx, end_idx))

    return matches


# ==============================
# 5. Hàm normalize_age_phrase
# ==============================

def _to_months(num_str: str, unit: str) -> int:
    """Chuyển '1', '1,5', '2.5' -> số tháng theo unit (tháng / tuổi / tuần)."""
    val = float(num_str.replace(",", "."))
    unit = unit.lower()

    if unit in MONTH_WORDS:
        return int(round(val))          # 1.5 tháng -> 2 tháng (hiếm gặp)
    elif unit in YEAR_WORDS:
        return int(round(val * 12))     # 1.5 tuổi -> 18 tháng
    elif unit in WEEK_WORDS:
        # 1 tuần ~ 7/30 tháng
        m = int(round(val * 7.0 / 30.0))
        return max(0, m)
    else:
        raise ValueError(f"Unit không hợp lệ: {unit}")


def normalize_age_phrase(raw: str) -> str | None:
    """
    Chuẩn hóa 1 cụm tuổi/tháng về canonical theo THÁNG.

    Kết quả:
      - '6-11M'    : 6–11 tháng
      - '0-12M'    : 0–12 tháng / dưới 12 tháng / sơ sinh đến 12 tháng
      - '12-36M'   : 1–3 tuổi
      - '6-60M'    : 6 tháng đến 5 tuổi
      - '0-18M'    : 0–1,5 tuổi
      - '6M+'      : từ 6 tháng / sau 6 tháng / 6 tháng trở lên
      - '30M+'     : từ 2.5 tuổi trở lên
      - '0M+'      : từ sơ sinh
    """

    if not raw:
        return None

    s = raw.strip().lower()

    # Chuẩn hóa 'tháng tuổi' -> 'tháng'
    s2 = s.replace("tháng tuổi", " tháng ")
    s2 = s2.replace("thang tuoi", " thang ")
    s2 = re.sub(r"\s+", " ", s2)

    # ==========================
    # 1) CASE 'sơ sinh'
    # ==========================
    if "sơ sinh" in s2:
        # 1.1. '(từ giai đoạn) sơ sinh ... đến ... N tháng/tuổi'
        m_nb_range = re.search(
            r"(?:từ\s+giai\s*đoạn\s+)?sơ\s*sinh"
            r".{0,30}?(?:đến|tới|to|-).{0,30}?"
            r"(\d{1,2}(?:[.,]\d+)?)\s*"
            r"(tháng|thang|month|months|tuổi|year|years)",
            s2,
        )
        if m_nb_range:
            n_str = m_nb_range.group(1)
            unit  = m_nb_range.group(2)
            end_m = _to_months(n_str, unit)
            if end_m <= 0:
                return None
            return f"0-{end_m}M"

        # 1.2. 'từ sơ sinh' -> 0M+
        m_nb_from0 = re.search(r"từ\s+sơ\s*sinh\b(?:\s*(trở lên|\+))?", s2)
        if m_nb_from0:
            return "0M+"

        # 1.3. 'sơ sinh từ/hơn/trên N tháng/tuổi' -> NM+
        m_nb_plus = re.search(
            r"sơ\s*sinh\s*(từ|hơn|trên|sau|>=|over|more than)\s*"
            r"(\d{1,2}(?:[.,]\d+)?)\s*"
            r"(tháng|thang|month|months|tuổi|year|years)"
            r"(?:\s*(trở lên|\+))?",
            s2,
        )
        if m_nb_plus:
            n_str = m_nb_plus.group(2)
            unit  = m_nb_plus.group(3)
            start_m = _to_months(n_str, unit)
            return f"{start_m}M+"

        # còn lại: 'sơ sinh' mô tả, không coi là mốc tuổi

    # ==========================
    # 2) CROSS RANGE đặc biệt '0-6 tháng ... đến ... 3 tuổi'
    # ==========================
    m_cross = re.search(
        r"(\d{1,2})\s*-\s*(\d{1,2})\s*"
        r"(tháng|thang|month|months)"
        r".{0,40}?(?:đến|tới|to)\s*.{0,40}?"
        r"(\d{1,2}(?:[.,]\d+)?)\s*(tuổi|year|years)",
        s2,
    )
    if m_cross:
        # Semantics: '0-6 tháng đến 3 tuổi' ~ 0 -> 3 tuổi
        y_end_str = m_cross.group(4)
        end_m = _to_months(y_end_str, "tuổi")
        return f"0-{end_m}M"

    # ==========================
    # 3) UNDER: 'dưới 12 tháng', 'under 2 years'
    # ==========================
    m_under = re.search(
        r"(dưới|under|<)\s*(\d{1,2}(?:[.,]\d+)?)\s*"
        r"(tháng|thang|month|months|tuổi|year|years|tuần|tuan|week|weeks)",
        s2,
    )
    if m_under:
        n_str = m_under.group(2)
        unit  = m_under.group(3)
        end_m = _to_months(n_str, unit)
        if end_m <= 0:
            return None
        return f"0-{end_m}M"

    # ==========================
    # 4) FROM / OVER / SAU: 'từ 1 tuổi', 'trên 1 tuổi', 'sau 6 tháng'
    #      -> XM+
    # ==========================
    m_from = re.search(
        r"(từ|trên|hơn|sau|>=|over|more than)\s*"
        r"(\d{1,2}(?:[.,]\d+)?)\s*"
        r"(tháng|thang|month|months|tuổi|year|years|tuần|tuan|week|weeks)"
        r"(?:\s*(trở lên|\+))?",
        s2,
    )
    if m_from:
        n_str = m_from.group(2)
        unit  = m_from.group(3)
        start_m = _to_months(n_str, unit)
        return f"{start_m}M+"

    # ==========================
    # 5) PLUS (hậu tố): '6 tháng tuổi trở lên', '2 tuổi trở lên'
    #      -> XM+
    # ==========================
    m_suffix_plus = re.search(
        r"(\d{1,2}(?:[.,]\d+)?)\s*"
        r"(tháng|thang|month|months|tuổi|year|years|tuần|tuan|week|weeks)\s*"
        r"(trở lên|\+)",
        s2,
    )
    if m_suffix_plus:
        n_str = m_suffix_plus.group(1)
        unit  = m_suffix_plus.group(2)
        start_m = _to_months(n_str, unit)
        return f"{start_m}M+"

    # ==========================
    # 6) Range 2 đơn vị: '6 tháng đến 5 tuổi', 'từ 1 tuần - 1 tuổi'
    # ==========================
    m_both = re.search(
        r"(?:từ\s*)?"
        r"(\d{1,2}(?:[.,]\d+)?)\s*"
        r"(tháng|thang|month|months|tuổi|year|years|tuần|tuan|week|weeks)\s*"
        r"(?:-|–|—|đến|to)\s*"
        r"(\d{1,2}(?:[.,]\d+)?)\s*"
        r"(tháng|thang|month|months|tuổi|year|years|tuần|tuan|week|weeks)",
        s2,
    )
    if m_both:
        a_str, u1, b_str, u2 = m_both.group(1), m_both.group(2), m_both.group(3), m_both.group(4)
        start_m = _to_months(a_str, u1)
        end_m   = _to_months(b_str, u2)
        if start_m > end_m:
            start_m, end_m = end_m, start_m
        return f"{start_m}-{end_m}M"

    # ==========================
    # 7) Range 1 đơn vị explicit: '0-12 tháng', '2-6 tuổi', '1-4 tuần'
    # ==========================
    m_range_unit = re.search(
        r"(\d{1,2}(?:[.,]\d+)?)\s*"
        r"(?:-|–|—|đến|to)\s*"
        r"(\d{1,2}(?:[.,]\d+)?)\s*"
        r"(tháng|thang|month|months|tuổi|year|years|tuần|tuan|week|weeks)",
        s2,
    )
    if m_range_unit:
        a_str = m_range_unit.group(1)
        b_str = m_range_unit.group(2)
        unit  = m_range_unit.group(3)
        start_m = _to_months(a_str, unit)
        end_m   = _to_months(b_str, unit)
        if start_m > end_m:
            start_m, end_m = end_m, start_m
        return f"{start_m}-{end_m}M"

    # ==========================
    # 8) Range "độ tuổi từ 2 đến 6." (mặc định đơn vị = tuổi)
    # ==========================
    m_ageword = re.search(
        r"độ\s*tuổi[^0-9]{0,20}"
        r"(?:từ\s*)?"
        r"(\d{1,2}(?:[.,]\d+)?)\s*(?:-|–|—|đến|to)\s*"
        r"(\d{1,2}(?:[.,]\d+)?)",
        s2,
    )
    if m_ageword:
        a_str = m_ageword.group(1)
        b_str = m_ageword.group(2)
        start_m = _to_months(a_str, "tuổi")
        end_m   = _to_months(b_str, "tuổi")
        if start_m > end_m:
            start_m, end_m = end_m, start_m
        return f"{start_m}-{end_m}M"

    # 9) Single còn lại -> bỏ (để tránh '3 tuổi' lẻ loi)
    return None

# ==============================
# 6. Hàm normalize_age_phrase_list
# ==============================

def _parse_canonical(c: str) -> tuple[int, int | None, bool]:
    """
    Trả (start_m, end_m, is_plus)
      - '6M'    -> (6, 6, False)
      - '6-36M' -> (6, 36, False)
      - '12M+'  -> (12, None, True)
    """
    if c.endswith("M+"):
        start = int(c[:-2])
        return start, None, True

    if not c.endswith("M"):
        raise ValueError(f"Canonical age phải kết thúc bằng 'M' hoặc 'M+': {c}")

    body = c[:-1]
    if "-" in body:
        a, b = body.split("-", 1)
        return int(a), int(b), False
    else:
        v = int(body)
        return v, v, False


def normalize_age_phrase_list(raw_list: list[str] | None) -> list[str]:
    if raw_list is None:
        return []

    # 1) Chuẩn hoá từng cụm
    canon: list[str] = []
    for raw in raw_list:
        norm = normalize_age_phrase(raw)
        if norm is not None:
            canon.append(norm)

    # 2) unique theo thứ tự
    uniq: list[str] = []
    for c in canon:
        if c not in uniq:
            uniq.append(c)

    if not uniq:
        return []

    # 3) loại các mốc bị "bao phủ" bởi 1 khoảng khác
    parsed = [_parse_canonical(c) for c in uniq]
    keep = [True] * len(uniq)

    for i, (si, ei, plus_i) in enumerate(parsed):
        if not keep[i]:
            continue
        for j, (sj, ej, plus_j) in enumerate(parsed):
            if i == j:
                continue

            # j bao phủ i?
            if plus_j:
                # KHÔNG cho khoảng 'sjM+' cover khoảng hữu hạn [si, ei]
                # => nếu i là finite, bỏ qua luôn
                if not plus_i:
                    continue

                # j là 'sjM+' -> [sj, +∞)
                if ei is None:
                    # cả 2 đều '+': giữ cái xuất hiện trước
                    if j < i:
                        keep[i] = False
                        break
                else:
                    if si >= sj:
                        keep[i] = False
                        break
            else:
                # j là khoảng hữu hạn [sj, ej]
                if ei is None:
                    # khoảng vô hạn không bị cover bởi hữu hạn
                    continue
                if sj <= si and ej >= ei:
                    keep[i] = False
                    break

    result = [c for c, k in zip(uniq, keep) if k]
    return result

In [150]:
df_unknown = (
    df_age
    .filter(pl.col("age_group_final") == "Không xác định")
    .with_columns(
        pl.concat_str(
            [
                pl.col("description").fill_null(""),
                pl.col("description_new").fill_null(""),
            ],
            separator=" "
        )
        .str.to_lowercase()
        .alias("desc_all")
    )
    .with_columns(
        pl.col("desc_all").map_elements(
            extract_age_phrases,
            return_dtype=pl.List(pl.Utf8)
        ).alias("age_phrases_raw")
    )
    .with_columns(
        pl.col("age_phrases_raw").map_elements(
            normalize_age_phrase_list,
            return_dtype=pl.List(pl.Utf8)
        ).alias("age_ranges_norm")
    )
)


In [151]:
# Lấy tất cả canonical ranges đã detect
target_norm_list = (
    df_unknown
    .select(pl.col("age_ranges_norm"))
    .explode("age_ranges_norm")
    .filter(pl.col("age_ranges_norm").is_not_null())
    .unique()
    .to_series()
    .to_list()
)

print("Số lượng target_norm:", len(target_norm_list))
target_norm_list

Số lượng target_norm: 148


['6-48M',
 '5-8M',
 '24-60M',
 '60M+',
 '12M+',
 '96-144M',
 '60-84M',
 '1-12M',
 '0-12M',
 '48-144M',
 '12-60M',
 '14-17M',
 '0-9M',
 '6-18M',
 '7-60M',
 '24-120M',
 '1-24M',
 '36-120M',
 '6-24M',
 '10M+',
 '9-11M',
 '7-24M',
 '24-108M',
 '72-144M',
 '7-8M',
 '9-12M',
 '4-30M',
 '4M+',
 '9-48M',
 '60-72M',
 '0-5M',
 '0M+',
 '0-4M',
 '12-24M',
 '0-48M',
 '2M+',
 '7M+',
 '0-144M',
 '48-72M',
 '0-1M',
 '36-84M',
 '36M+',
 '9-20M',
 '3-18M',
 '18M+',
 '48-60M',
 '6-60M',
 '108-132M',
 '1M+',
 '84M+',
 '60-132M',
 '0-25M',
 '5M+',
 '12-120M',
 '9M+',
 '12-192M',
 '0-36M',
 '24-96M',
 '0-11M',
 '18-24M',
 '1-2M',
 '4-24M',
 '18-21M',
 '36-72M',
 '36-288M',
 '6-144M',
 '0-72M',
 '12-14M',
 '13-16M',
 '0-24M',
 '24-30M',
 '12-108M',
 '144-216M',
 '6-8M',
 '6-72M',
 '6-9M',
 '12-96M',
 '84-120M',
 '144M+',
 '12-48M',
 '72-108M',
 '4-72M',
 '17-18M',
 '9-72M',
 '19-25M',
 '108-144M',
 '9-24M',
 '4-6M',
 '0-3M',
 '120M+',
 '17M+',
 '3-24M',
 '0-84M',
 '3-6M',
 '36-48M',
 '36-192M',
 '108-120M',


Các trường hợp age_group bắt được -> đã check kĩ

In [152]:
import textwrap

pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_rows(50)

# Hàm wrap text 175 ký tự
def wrap175(text):
    if text is None:
        return ""
    return textwrap.fill(text, width=175)

with open("output_age_norm.txt", "w", encoding="utf-8") as f:

    for tg in target_norm_list:
        f.write(f"\n=== VÍ DỤ CHO TARGET_NORM = {tg} ===\n\n")
        target_norm = tg

        examples = (
            df_unknown
            .filter(pl.col("age_ranges_norm").list.contains(target_norm))
            .select([
                "item_id",
                "age_group_final",
                "age_phrases_raw",
                "age_ranges_norm",
                "description",
                "description_new",
            ])
            .head(20)
        )

        for row in examples.iter_rows(named=True):
            f.write("======================================\n")
            f.write(f"item_id: {row['item_id']}\n")
            f.write(f"age_group_final: {row['age_group_final']}\n")
            f.write(f"age_phrases_raw: {row['age_phrases_raw']}\n")
            f.write(f"age_ranges_norm: {row['age_ranges_norm']}\n\n")

            f.write("description:\n")
            f.write(wrap175(str(row["description"])) + "\n\n")

            f.write("description_new:\n")
            f.write(wrap175(str(row["description_new"])) + "\n\n")

print("Đã xuất file: output_age_norm.txt")

Đã xuất file: output_age_norm.txt


Fill các trường hợp `age_group` bắt được từ 2 description (chắc chắn)

In [153]:
def build_age_group_value(age_ranges_norm, age_phrases_raw):
    """
    age_ranges_norm: list các canonical như ['0-6M', '6-12M'] hoặc ['6M+'] …
    age_phrases_raw: list các câu raw đã match từ 2 description.

    Trả về:
      - None: nếu không có mốc nào (list rỗng / None)
      - '6-12M': nếu chỉ có 1 mốc
      - dict {...}: nếu >= 2 mốc
    """
    if not age_ranges_norm:
        return None

    # Chỉ 1 mốc -> fill thẳng là string
    if len(age_ranges_norm) == 1:
        return age_ranges_norm[0]

    # Nhiều mốc -> trả dict để bạn soi/giữ full info
    return {
        "source": "description",
        "ranges_norm": age_ranges_norm,
        "phrases_raw": age_phrases_raw,
    }

In [154]:
import polars as pl

# 1.1. Gộp description + description_new
df_age = df_age.with_columns(
    pl.concat_str(
        [
            pl.col("description").fill_null(""),
            pl.col("description_new").fill_null(""),
        ],
        separator=" "
    )
    .str.to_lowercase()
    .alias("desc_all")
)

# 1.2. Bắt các cụm tuổi thô từ desc_all
df_age = df_age.with_columns(
    pl.col("desc_all")
    .map_elements(
        extract_age_phrases,              # hàm Python đã viết
        return_dtype=pl.List(pl.Utf8),
    )
    .alias("age_phrases_raw")
)

# 1.3. Chuẩn hoá về dạng canonical tháng ('0-12M', '6M+', ...)
df_age = df_age.with_columns(
    pl.col("age_phrases_raw")
    .map_elements(
        normalize_age_phrase_list,        # hàm Python đã viết
        return_dtype=pl.List(pl.Utf8),
    )
    .alias("age_ranges_norm")
)

In [155]:
df_age.select(["item_id", "age_group_final", "age_phrases_raw", "age_ranges_norm"]).head(5)

item_id,age_group_final,age_phrases_raw,age_ranges_norm
str,str,list[str],list[str]
"""0502020000004""","""Không xác định""","[""trên 9 tháng"", ""trên 9 tháng"", ""trên 9 tháng tuổi""]","[""9M+""]"
"""0010290040150""","""Từ 3Y""",[],[]
"""0008010000015""","""0-1Y""",[],[]
"""0020010000094""","""Không xác định""",[],[]
"""0020010000098""","""Không xác định""",[],[]


In [156]:
df_age = df_age.with_columns(
    pl.struct(["age_ranges_norm", "age_phrases_raw"])
    .map_elements(
        lambda row: build_age_group_value(
            row["age_ranges_norm"],
            row["age_phrases_raw"],
        ),
        return_dtype=pl.Object,   # để chứa được cả str và dict
    )
    .alias("age_group_from_desc")
)

In [157]:
import json
import polars as pl

df_age = df_age.with_columns(
    pl.col("age_group_from_desc")
    .map_elements(
        lambda v: (
            None
            if v is None
            else (v if isinstance(v, str) else json.dumps(v, ensure_ascii=False))
        ),
        return_dtype=pl.Utf8,
    )
    .alias("age_group_from_desc_str")
)

In [158]:
df_age = df_age.with_columns(
    pl.when(
        (pl.col("age_group_final") == "Không xác định")
        & pl.col("age_group_from_desc_str").is_not_null()
    )
    .then(pl.col("age_group_from_desc_str"))
    .otherwise(pl.col("age_group_final"))
    .alias("age_group_final")
)

In [159]:
pl.Config.set_fmt_str_lengths(50)
pl.Config.set_tbl_rows(50)

polars.config.Config

In [160]:
df_age = df_age.drop([
    "age_phrases_raw",
    "age_ranges_norm",
    "age_group_from_desc",
    "age_group_from_desc_str",
])


In [161]:
total = df_age.height

unknown = (
    df_age
    .filter(pl.col("age_group_final") == "Không xác định")
    .height
)

ratio = unknown / total * 100

print("Tổng số dòng:", total)
print("Số dòng còn 'Không xác định':", unknown)
print(f"Tỷ lệ còn lại: {ratio:.2f}%")

Tổng số dòng: 27323
Số dòng còn 'Không xác định': 11784
Tỷ lệ còn lại: 43.13%


#### Ý tưởng 2:

# Task 3: Phân tích tương đồng và xác định xem các thuộc tính tương tự nhau. Từ đó loại bỏ đặc trưng thừa.

# Task 4: Chuẩn hóa dữ liệu (nếu có), biến đổi dữ liệu

# Task 5: Nhóm hãy suy nghĩ xem, với bài toán dự đoán mua hàng, ta có thể tạo mới những đặc trưng nào. Sau đó tiến hành rút trích thêm các đặc trưng. Task này rất quan trọng vì ảnh hưởng hiệu quả của hệ thống.